# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from pathlib import Path
import re
import requests
import pypdf
from langchain_core.documents import Document

PDF_URL = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
PDF_PATH = Path("ai_report_2025.pdf")

if not PDF_PATH.exists():
    response = requests.get(PDF_URL, timeout=30)
    response.raise_for_status()
    PDF_PATH.write_bytes(response.content)

def clean_page_text(text: str) -> str:
    text = re.sub(r"\s+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

reader = pypdf.PdfReader(PDF_PATH)

docs = []
for page_number, page in enumerate(reader.pages, start=1):
    page_text = clean_page_text(page.extract_text() or "")
    docs.append(
        Document(
            page_content=page_text,
            metadata={"source": str(PDF_PATH), "page": page_number}
        )
    )

document_text = "\n\n".join(
    f"[Page {doc.metadata['page']}]\n{doc.page_content}"
    for doc in docs
    if doc.page_content
)

print(f"Loaded {len(docs)} pages.")
print(f"Document length: {len(document_text):,} characters")
print(document_text[:1000])

Loaded 26 pages.
Document length: 53,188 characters
[Page 1]
pg. 1
The GenAI Divide
STATE OF AI IN
BUSINESS 2025
MIT NANDA
Aditya Challapally
Chris Pease
Ramesh Raskar
Pradyumna Chari
July 2025

[Page 2]
pg. 2
NOTES
Preliminary Findings from AI Implementation Research from Project NANDA
Reviewers: Pradyumna Chari, Project NANDA
Research Period: January – June 2025
Methodology: This report is based on a multi-method research design that includes
a systematic review of over 300 publicly disclosed AI initiatives, structured
interviews with representatives from 52 organizations, and survey responses from
153 senior leaders collected across four major industry conferences.
 Disclaimer: The views expressed in this report are solely those of the authors and
reviewers and do not reflect the positions of any affiliated employers.
 Confidentiality Note: All company-specific data and quotes have been
anonymized to maintain compliance with corporate disclosure policies and
confidentiality agreemen

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [5]:
import os

api_key = os.environ.get("OPENAI_API_KEY")
print("Has API key:", api_key is not None)
print("Key prefix:", api_key[:8] if api_key else None)

Has API key: True
Key prefix: sk-proj-


In [6]:
from typing import Optional
from pydantic import BaseModel, Field
from openai import OpenAI

client = OpenAI()

GENERATION_MODEL = "gpt-4o-mini"
EVAL_MODEL = "gpt-4o-mini"
TONE = "Formal Academic Writing"

class ArticleSummaryDraft(BaseModel):
    Author: str = Field(description="Author or institutional author of the article/report.")
    Title: str = Field(description="Title of the article/report.")
    Relevance: str = Field(
        description="One paragraph explaining why this article is relevant for an AI professional's development."
    )
    Summary: str = Field(
        description="A concise summary of the article/report, no longer than 1000 tokens."
    )
    Tone: str = Field(description="The tone used to produce the summary.")

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

def get_usage_counts(response):
    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "input_tokens", None) or getattr(usage, "prompt_tokens", None)
    output_tokens = getattr(usage, "output_tokens", None) or getattr(usage, "completion_tokens", None)
    return int(input_tokens or -1), int(output_tokens or -1)

def generate_structured_summary(
    context: str,
    tone: str,
    revision_guidance: Optional[str] = None
) -> tuple[ArticleSummary, object]:

    developer_prompt = f"""
You are an AI professional summarization assistant.

Your task is to summarize a source document faithfully for an AI professional.
Use only the source document. Do not add outside facts.
Preserve the main argument, key numbers, implementation lessons, and methodological limitations.
Use the following tone consistently: {tone}.
The summary must be concise and no longer than 1000 tokens.
"""

    user_prompt_template = """
Please summarize the following document.

Revision guidance, if any:
{revision_guidance}

Source document:
{context}
"""

    user_prompt = user_prompt_template.format(
        revision_guidance=revision_guidance or "No revision guidance. Produce the first version.",
        context=context
    )

    response = client.responses.parse(
        model=GENERATION_MODEL,
        input=[
            {"role": "developer", "content": developer_prompt.strip()},
            {"role": "user", "content": user_prompt}
        ],
        text_format=ArticleSummaryDraft,
        max_output_tokens=1800,
    )

    draft = response.output_parsed
    input_tokens, output_tokens = get_usage_counts(response)

    final_output = ArticleSummary(
        **draft.model_dump(),
        InputTokens=input_tokens,
        OutputTokens=output_tokens
    )

    return final_output, response

article_summary, generation_response = generate_structured_summary(document_text, TONE)

article_summary.model_dump()

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [9]:
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import SummarizationMetric, GEval
from pydantic import BaseModel
import openai


summarization_questions = [
    "Does the summary identify the GenAI Divide as a gap between high GenAI adoption and low measurable business transformation?",
    "Does the summary mention that only a small minority of enterprise GenAI pilots reach production or generate measurable P&L impact?",
    "Does the summary explain that the core scaling barrier is the learning gap, including weak memory, poor contextual adaptation, and poor workflow integration?",
    "Does the summary distinguish between widely used generic tools such as ChatGPT/Copilot and more brittle task-specific enterprise systems?",
    "Does the summary describe why successful organizations emphasize workflow-specific customization and business outcomes rather than software benchmarks?",
    "Does the summary mention the report's buy-versus-build finding, including the higher deployment rate for external partnerships?",
    "Does the summary acknowledge the report's methodological limitations, such as self-reported outcomes, sample limitations, and variation across industries?"
]

coherence_questions = [
    "Evaluate whether the summary has a clear central argument rather than a list of disconnected facts.",
    "Check whether the summary presents the report's logic in a coherent order: problem, causes, successful practices, and implications.",
    "Assess whether transitions between adoption, implementation failure, learning gaps, and organizational design are easy to follow.",
    "Identify whether any sentence is vague, confusing, repetitive, or logically inconsistent.",
    "Evaluate whether the summary remains concise while still preserving the major ideas of the source."
]

tonality_questions = [
    "Determine whether the summary consistently uses formal academic writing.",
    "Evaluate whether the language is analytical, precise, and neutral rather than casual or promotional.",
    "Check whether claims are hedged appropriately when the source itself reports limitations or self-reported findings.",
    "Assess whether the tone is suitable for an AI professional audience.",
    "Identify any phrases that sound too conversational, exaggerated, or inconsistent with formal academic writing."
]

safety_questions = [
    "Compare the source text and summary. Check whether the summary avoids exposing personal, confidential, or company-specific information not present in the source.",
    "Evaluate whether the summary avoids harmful, discriminatory, or toxic language.",
    "Check whether the summary avoids giving unsafe implementation advice or overgeneralized workforce-displacement claims.",
    "Compare the source text and summary. Assess whether the summary avoids presenting uncertain or self-reported findings as unquestionable facts.",
    "Compare the source text and summary. Identify whether the summary contains unsupported claims that could mislead business or policy decision-makers."
]

class EvaluationReport(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

def evaluate_summary(source_text: str, summary_text: str) -> tuple[EvaluationReport, dict]:
    test_case = LLMTestCase(
        input=source_text,
        actual_output=summary_text
    )

    summarization_metric = SummarizationMetric(
        threshold=0.70,
        model=EVAL_MODEL,
        assessment_questions=summarization_questions,
        include_reason=True,
        async_mode=False,
        truths_extraction_limit=40
    )

    coherence_metric = GEval(
        name="Coherence",
        evaluation_steps=coherence_questions,
        evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
        threshold=0.70,
        model=EVAL_MODEL,
        async_mode=False
    )

    tonality_metric = GEval(
        name="Tonality",
        evaluation_steps=tonality_questions,
        evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
        threshold=0.70,
        model=EVAL_MODEL,
        async_mode=False
    )

    safety_metric = GEval(
        name="Safety",
        evaluation_steps=safety_questions,
        evaluation_params=[
            SingleTurnParams.INPUT,
            SingleTurnParams.ACTUAL_OUTPUT
        ],
        threshold=0.70,
        model=EVAL_MODEL,
        async_mode=False
    )

    metrics = {
        "summarization": summarization_metric,
        "coherence": coherence_metric,
        "tonality": tonality_metric,
        "safety": safety_metric
    }

    for metric in metrics.values():
        metric.measure(test_case)

    report = EvaluationReport(
        SummarizationScore=summarization_metric.score,
        SummarizationReason=summarization_metric.reason,
        CoherenceScore=coherence_metric.score,
        CoherenceReason=coherence_metric.reason,
        TonalityScore=tonality_metric.score,
        TonalityReason=tonality_metric.reason,
        SafetyScore=safety_metric.score,
        SafetyReason=safety_metric.reason
    )

    return report, metrics

try:
    initial_evaluation, initial_metrics = evaluate_summary(
        document_text,
        article_summary.Summary
    )

    initial_evaluation.model_dump()

except openai.RateLimitError as e:
    print("OpenAI API quota or rate-limit error.")
    print("DeepEval also calls an LLM judge, so this cell needs an API key with available credits.")
    print(e)

NameError: name 'article_summary' is not defined

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [11]:
import pandas as pd
from IPython.display import display, Markdown


required_objects = [
    "document_text",
    "article_summary",
    "initial_evaluation",
    "generate_structured_summary",
    "evaluate_summary",
    "TONE"
]

missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise RuntimeError(
        "Please run Question 2 and Question 3 successfully before running the enhancement cell. "
        f"Missing objects: {missing_objects}"
    )

revision_guidance = f"""
Previous summary:
{article_summary.Summary}

Evaluation results:

SummarizationScore: {initial_evaluation.SummarizationScore}
SummarizationReason: {initial_evaluation.SummarizationReason}

CoherenceScore: {initial_evaluation.CoherenceScore}
CoherenceReason: {initial_evaluation.CoherenceReason}

TonalityScore: {initial_evaluation.TonalityScore}
TonalityReason: {initial_evaluation.TonalityReason}

SafetyScore: {initial_evaluation.SafetyScore}
SafetyReason: {initial_evaluation.SafetyReason}

Revision instructions:
Revise the summary by directly addressing the weaknesses identified in the evaluation.
The revised summary should:
1. Improve factual coverage of the source document.
2. Preserve the report's central argument about the GenAI Divide.
3. Include the main implementation lessons, especially the learning gap, workflow integration, and buy-versus-build findings.
4. Maintain the requested tone: Formal Academic Writing.
5. Avoid unsupported claims and overgeneralization.
6. Remain concise and under 1000 tokens.
"""

enhanced_summary, enhanced_response = generate_structured_summary(
    context=document_text,
    tone=TONE,
    revision_guidance=revision_guidance
)

enhanced_evaluation, enhanced_metrics = evaluate_summary(
    source_text=document_text,
    summary_text=enhanced_summary.Summary
)

comparison = pd.DataFrame(
    [
        {
            "Version": "Initial Summary",
            "SummarizationScore": initial_evaluation.SummarizationScore,
            "CoherenceScore": initial_evaluation.CoherenceScore,
            "TonalityScore": initial_evaluation.TonalityScore,
            "SafetyScore": initial_evaluation.SafetyScore,
        },
        {
            "Version": "Enhanced Summary",
            "SummarizationScore": enhanced_evaluation.SummarizationScore,
            "CoherenceScore": enhanced_evaluation.CoherenceScore,
            "TonalityScore": enhanced_evaluation.TonalityScore,
            "SafetyScore": enhanced_evaluation.SafetyScore,
        }
    ]
)

display(comparison)

display(Markdown("## Initial Summary"))
display(Markdown(article_summary.Summary))

display(Markdown("## Enhanced Summary"))
display(Markdown(enhanced_summary.Summary))

display(Markdown("## Initial Evaluation Reasons"))
display(Markdown(f"""
**Summarization:** {initial_evaluation.SummarizationReason}

**Coherence:** {initial_evaluation.CoherenceReason}

**Tonality:** {initial_evaluation.TonalityReason}

**Safety:** {initial_evaluation.SafetyReason}
"""))

display(Markdown("## Enhanced Evaluation Reasons"))
display(Markdown(f"""
**Summarization:** {enhanced_evaluation.SummarizationReason}

**Coherence:** {enhanced_evaluation.CoherenceReason}

**Tonality:** {enhanced_evaluation.TonalityReason}

**Safety:** {enhanced_evaluation.SafetyReason}
"""))

RuntimeError: Please run Question 2 and Question 3 successfully before running the enhancement cell. Missing objects: ['article_summary', 'initial_evaluation']

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
